# Time series forecasting with ARIMA

In this notebook, we demonstrate how to:
- prepare time series data for training an ARIMA time series forecasting model
- implement a simple ARIMA model to forecast the next HORIZON steps ahead (time *t+1* through *t+HORIZON*) in the time series
- evaluate the model 


The data in this example is taken from the GEFCom2014 forecasting competition<sup>1</sup>. It consists of 3 years of hourly electricity load and temperature values between 2012 and 2014. The task is to forecast future values of electricity load. In this example, we show how to forecast one time step ahead, using historical load data only.

<sup>1</sup>Tao Hong, Pierre Pinson, Shu Fan, Hamidreza Zareipour, Alberto Troccoli and Rob J. Hyndman, "Probabilistic energy forecasting: Global Energy Forecasting Competition 2014 and beyond", International Journal of Forecasting, vol.32, no.3, pp 896-913, July-September, 2016.

In [1]:
pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.3 MB 7.6 MB/s eta 0:00:02
   ------------ --------------------------- 3.4/11.3 MB 10.3 MB/s eta 0:00:01
   -------------------- ------------------- 5.8/11.3 MB 10.8 MB/s eta 0:00:01
   ---------------------------- ----------- 8.1/11.3 MB 11.1 MB/s eta 0:00:01
   ---------------------------------- ----- 9.7/11.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 10.4 MB/s  0:00:01
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.4/36.6 MB 11.6 MB/s eta 0:00:03
   ----- ---------------------------------- 4.7/36.6 MB 11.7 MB/s eta 0:00:03
   ------- -------------------------------- 7.1/36.6 MB 11.7 MB/s eta 0:00:03
   ---------- ----------------------------- 9.7/36.6 MB 11.7 MB/s eta 0:00:03
   ------------- -------------------------- 12.1/36.6 MB 11.7 MB/s eta 0:00:03
   -


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
import math

from pandas.plotting import autocorrelation_plot
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from common.utils import load_data, mape
from IPython.display import Image

%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format
np.set_printoptions(precision=2)
warnings.filterwarnings('ignore')

In [12]:
import os
import matplotlib
import numpy
import pandas
import statsmodels
import sklearn
import IPython

print("All main libraries are installed!")
print("Matplotlib:", matplotlib.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Statsmodels:", statsmodels.__version__)

All main libraries are installed!
Matplotlib: 3.11.2
NumPy: 2.4.0
Pandas: 2.3.3
Scikit-learn: 1.9.1
Statsmodels: 0.15.0


In [13]:
energy = load_data('./data')[['load']]
energy.head(10)

,load
2012-01-01 00:00:00,"2,698.00"
2012-01-01 01:00:00,"2,558.00"
2012-01-01 02:00:00,"2,444.00"
2012-01-01 03:00:00,"2,402.00"
2012-01-01 04:00:00,"2,403.00"
2012-01-01 05:00:00,"2,453.00"
2012-01-01 06:00:00,"2,560.00"
2012-01-01 07:00:00,"2,719.00"
2012-01-01 08:00:00,"2,916.00"
2012-01-01 09:00:00,"3,105.00"
